# <font color="#418FDE" size="6.5" uppercase>**Autoencoder und Generierung**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Unterscheiden diskriminative und generative Aufgaben anhand kleiner Datenbeispiele. 
- Trainieren kleine Autoencoder zur Rekonstruktion von Digits- oder MNIST-Teilmengen. 
- Analysieren Rekonstruktionsfehler, Denoising, latente Interpolation und Grenzen generativer Ansätze. 


## **1. Autoencoder Idee**

### **1.1. Diskriminativ oder generativ**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_01.jpg?v=1784819130" width="250">



>* Diskriminative Modelle treffen Entscheidungen über Daten
>* Sie lernen Merkmale zur Klassenunterscheidung

>* Generative Modelle lernen plausible Datenmuster.
>* Sie ergänzen, bereinigen und erzeugen Beispiele.

>* Autoencoder rekonstruieren statt direkt zu klassifizieren
>* Bewertung über Ähnlichkeit, Plausibilität und gelernte Strukturen



In [ ]:
#@title Python-Code - Diskriminativ oder generativ

# Dieses Beispiel vergleicht zwei Lernaufgaben.
# Diskriminativ bedeutet entscheiden, generativ bedeutet erzeugen.
# Die Ausgabe zeigt passende Fragen und Bewertungen.

import numpy as np
import pandas as pd

# Kleine Bildmerkmale beschreiben vereinfachte handgeschriebene Ziffern.
data = pd.DataFrame(
    {
        "curves": [2, 3, 1, 4, 2, 5],
        "holes": [0, 2, 0, 2, 1, 2],
        "label": [3, 8, 1, 8, 6, 8],
    }
)

# Eine diskriminative Aufgabe fragt nach dem richtigen Label.
discriminative_question = "Welche Ziffer ist das?"
discriminative_answer = int(data.loc[1, "label"])

# Eine generative Aufgabe fragt nach plausiblen Datenmustern.
feature_means = data[["curves", "holes"]].mean()
generated_digit = np.rint(feature_means).astype(int)

# Rekonstruktion bewertet Ähnlichkeit statt Klassenrichtigkeit.
original_features = data.loc[1, ["curves", "holes"]].to_numpy()
reconstructed_features = generated_digit.to_numpy()
reconstruction_error = np.mean((original_features - reconstructed_features) ** 2)

# Die Tabelle macht Eingaben, Labels und Merkmale sichtbar.
print("Kleines Datenbeispiel mit Merkmalen und Labels:")
print(data.head(3).to_string(index=False))
print("Diskriminative Frage: " + discriminative_question)
print("Diskriminative Antwort für Zeile 2: " + str(discriminative_answer))
print("Generative Idee: erzeuge plausible Merkmale aus gelernten Mustern.")
print("Erzeugte Merkmale: curves=" + str(generated_digit["curves"]) + ", holes=" + str(generated_digit["holes"]))
print("Rekonstruktionsfehler für Zeile 2: " + str(round(reconstruction_error, 2)))



### **1.2. Encoder und Decoder**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_02.jpg?v=1784819132" width="250">



>* Encoder verdichtet Eingaben zu wichtigen Merkmalen
>* Decoder rekonstruiert daraus ähnliche Ausgaben

>* Diskriminative Modelle entscheiden zwischen Klassen
>* Autoencoder rekonstruieren wesentliche Datenmuster

>* Encoder und Decoder lernen gemeinsam.
>* Rekonstruktionen bleiben an Trainingsmuster gebunden.



In [ ]:
#@title Python-Code - Encoder und Decoder

# Dieses Beispiel zeigt Encoder und Decoder.
# Wir komprimieren Ziffernbilder in zwei Zahlen.
# Die Grafik vergleicht Original und Rekonstruktion.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import sklearn

# Die Digits-Daten sind kleine Graustufenbilder.
digits = load_digits()
images = digits.images
labels = digits.target

# Wir nutzen nur wenige Beispiele für Übersichtlichkeit.
selected_indices = np.array([0, 1, 2, 3, 4, 5])
selected_images = images[selected_indices]
selected_labels = labels[selected_indices]

# Jedes Bild wird zu einer Zeile mit 64 Pixelwerten.
flat_images = selected_images.reshape(len(selected_images), -1)
all_flat_images = images.reshape(len(images), -1)

# Der Encoder lernt eine zweidimensionale Kurzbeschreibung.
encoder_decoder = PCA(n_components=2, random_state=42)
encoder_decoder.fit(all_flat_images)

# Transform ist der Encoder, inverse_transform der Decoder.
latent_codes = encoder_decoder.transform(flat_images)
reconstructed_flat = encoder_decoder.inverse_transform(latent_codes)
reconstructed_images = reconstructed_flat.reshape(selected_images.shape)

# Der Fehler misst, wie viel Information verloren ging.
reconstruction_error = mean_squared_error(flat_images, reconstructed_flat)
explained = encoder_decoder.explained_variance_ratio_.sum()

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Latente Dimensionen: 64 Pixel -> 2 Zahlen")
print(f"Erklärte Varianz im Encoder: {explained:.2f}")
print(f"Mittlerer Rekonstruktionsfehler: {reconstruction_error:.2f}")
print(f"Beispiel-Code für Ziffer {selected_labels[0]}: {latent_codes[0].round(2)}")

# Eine Achse zeigt Originale oben und Rekonstruktionen unten.
comparison = np.vstack([selected_images, reconstructed_images])
separator = np.ones((1, comparison.shape[1], comparison.shape[2])) * 16
comparison = np.vstack([selected_images, separator, reconstructed_images])

# Die Bildmatrix macht den Informationsverlust sichtbar.
wide_image = np.hstack(comparison)
fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(wide_image, cmap="gray", vmin=0, vmax=16)
ax.set_title("Oben: Originale, unten: Decoder-Rekonstruktionen")
ax.set_xlabel("Ziffernbeispiele nebeneinander")
ax.set_ylabel("Originale und Rekonstruktionen")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **1.3. Latente Darstellung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_01_03.jpg?v=1784819134" width="250">



>* Latente Codes speichern wichtige Merkmale kompakt
>* Decoder macht verborgene Strukturen wieder sichtbar

>* Diskriminativ: Klassen trennen mit entscheidenden Merkmalen
>* Generativ: Datenvielfalt für plausible Beispiele erfassen

>* Ähnliche Eingaben liegen im latenten Raum nah
>* Schlechte Struktur erzeugt unklare Rekonstruktionen



## **2. Rekonstruktion trainieren**

### **2.1. Dichter Autoencoder**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_01.jpg?v=1784819136" width="250">



>* Encoder komprimiert Bilder in latente Merkmale
>* Decoder rekonstruiert Bilder ohne Klassenlabels

>* Dichte Schichten verbinden alle Einheiten vollständig
>* Engpass erzwingt wichtige Muster statt Kopieren

>* Eingabe und Ziel sind dasselbe Bild
>* Rekonstruktionsfehler zeigen gelernte kompakte Muster



In [ ]:
#@title Python-Code - Dichter Autoencoder

# Wir trainieren einen kleinen dichten Autoencoder.
# Der Engpass erzwingt eine kompakte Zifferndarstellung.
# Rekonstruktionsfehler zeigen den Lernerfolg sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images

# Wir prüfen die erwartete Bildform vor dem Training.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Bilder.")

# Pixelwerte werden für neuronale Netze auf null bis eins skaliert.
X = images.reshape(len(images), 64).astype(float) / 16.0

# Eingabe und Ziel sind beim Autoencoder identisch.
X_train, X_test = train_test_split(
    X,
    test_size=0.25,
    random_state=42,
)

# Ein kleines Netz mit Engpass rekonstruiert die Pixelwerte.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 8, 32),
    activation="relu",
    solver="adam",
    max_iter=120,
    random_state=42,
)

# Das Modell lernt, seine Eingaben wieder auszugeben.
autoencoder.fit(X_train, X_train)

# Rekonstruktionen werden auf ungültige Pixelbereiche begrenzt.
reconstructed = autoencoder.predict(X_test)
reconstructed = np.clip(reconstructed, 0.0, 1.0)

# Der Fehler misst den durchschnittlichen Pixelunterschied.
train_reconstructed = np.clip(autoencoder.predict(X_train), 0.0, 1.0)
train_error = mean_squared_error(X_train, train_reconstructed)
test_error = mean_squared_error(X_test, reconstructed)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsfehler MSE: {train_error:.4f}")
print(f"Testfehler MSE: {test_error:.4f}")

# Ein Bildpaar macht die Rekonstruktion direkt vergleichbar.
original_image = X_test[0].reshape(8, 8)
reconstructed_image = reconstructed[0].reshape(8, 8)
comparison = np.hstack([original_image, reconstructed_image])

fig, ax = plt.subplots(figsize=(5, 2.5))
ax.imshow(comparison, cmap="gray", vmin=0, vmax=1)
ax.set_title("Original links, Rekonstruktion rechts")
ax.set_xlabel("Pixelspalten")
ax.set_ylabel("Pixelzeilen")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **2.2. Rekonstruktionsverlust messen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_02.jpg?v=1784819138" width="250">



>* Verlust misst Unterschiede zur Eingabe
>* Gute Rekonstruktion bewahrt wichtige Bildinformationen

>* Durchschnittsverlust über viele Bilder betrachten
>* Trainings- und Validierungsverlust vergleichen

>* Niedriger Verlust garantiert keine gute Rekonstruktion
>* Verlust immer mit Bildern gemeinsam prüfen



In [ ]:
#@title Python-Code - Rekonstruktionsverlust messen

# Wir messen Rekonstruktionsverlust bei kleinen Ziffernbildern.
# PCA dient hier als einfacher Autoencoder.
# Niedriger Verlust bedeutet ähnlichere rekonstruierte Bilder.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Die Ziffernbilder sollten 8 mal 8 Pixel haben.")

# Pixelwerte werden auf den Bereich null bis eins skaliert.
flat_images = images.reshape(len(images), -1) / 16.0

# Trainings- und Validierungsdaten bleiben sauber getrennt.
train_images, valid_images = train_test_split(
    flat_images,
    test_size=0.25,
    random_state=42,
)

# PCA komprimiert und rekonstruiert wie ein linearer Autoencoder.
latent_dimensions = [2, 8, 16, 32]
train_losses = []
valid_losses = []

# Für jede Kompression messen wir den mittleren Pixelverlust.
for latent_dim in latent_dimensions:
    autoencoder = PCA(n_components=latent_dim, random_state=42)
    train_latent = autoencoder.fit_transform(train_images)
    train_reconstructed = autoencoder.inverse_transform(train_latent)

    valid_latent = autoencoder.transform(valid_images)
    valid_reconstructed = autoencoder.inverse_transform(valid_latent)

    train_loss = mean_squared_error(train_images, train_reconstructed)
    valid_loss = mean_squared_error(valid_images, valid_reconstructed)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

# Kurze Zahlen helfen beim Lesen der Kurve.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsbilder: {len(train_images)}, Validierungsbilder: {len(valid_images)}")
print(f"Bester Validierungsverlust: {min(valid_losses):.4f}")

# Die Kurve zeigt den Verlust bei stärkerer oder schwächerer Kompression.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(latent_dimensions, train_losses, marker="o", label="Training")
ax.plot(latent_dimensions, valid_losses, marker="o", label="Validierung")
ax.set_title("Rekonstruktionsverlust eines einfachen Autoencoders")
ax.set_xlabel("Dimension des latenten Raums")
ax.set_ylabel("Mittlerer quadratischer Pixelverlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.3. Originale und Rekonstruktionen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_02_03.jpg?v=1784819140" width="250">



>* Autoencoder komprimieren Bilder und rekonstruieren sie
>* Vergleiche zeigen gelernte Formen und Details

>* Fehler zeigen verlorene Details
>* Bilder ergänzen den Rekonstruktionsverlust

>* Rekonstruktionen bleiben eingabenah, nicht frei generativ
>* Mehrere Beispiele zeigen Lernen oder Auswendiglernen



In [ ]:
#@title Python-Code - Originale und Rekonstruktionen

# Wir vergleichen Originale mit Autoencoder-Rekonstruktionen.
# Ein kleiner MLP lernt komprimierte Ziffernbilder.
# Die Grafik zeigt typische Rekonstruktionsfehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Die Digits-Daten sind kleine Graustufenbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Wir prüfen die erwartete Bildform für sichere Darstellung.
if images.shape[1:] != (8, 8):
    raise ValueError("Die Digits-Bilder sollten 8 mal 8 Pixel haben.")

# Pixelwerte werden auf den Bereich null bis eins skaliert.
features = images.reshape(len(images), -1) / 16.0

# Der Autoencoder soll Eingaben als Ziel wiederherstellen.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=42, stratify=labels
)

# Eine kleine Engstelle erzwingt eine komprimierte Darstellung.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 4, 16), activation="relu", max_iter=300,
    random_state=42, solver="adam", learning_rate_init=0.01
)

# Das Modell lernt, jedes Bild aus sich selbst zu rekonstruieren.
autoencoder.fit(X_train, X_train)

# Rekonstruktionen werden auf ungesehene Testbilder angewendet.
reconstructed = autoencoder.predict(X_test)
reconstructed = np.clip(reconstructed, 0.0, 1.0)

# Der mittlere quadratische Fehler misst die Rekonstruktionsqualität.
mse = mean_squared_error(X_test, reconstructed)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Test-Rekonstruktionsfehler MSE: {mse:.4f}")
print("Oben: Originale, unten: Rekonstruktionen derselben Ziffern.")

# Wir wählen sechs feste Beispiele für einen fairen Vergleich.
example_indices = np.arange(6)
original_row = X_test[example_indices].reshape(6, 8, 8)
reconstructed_row = reconstructed[example_indices].reshape(6, 8, 8)

# Beide Reihen werden zu einem einzigen Bild zusammengesetzt.
comparison_image = np.zeros((17, 6 * 8))
comparison_image[:8, :] = np.hstack(original_row)
comparison_image[9:17, :] = np.hstack(reconstructed_row)

# Eine Achse genügt für den direkten visuellen Vergleich.
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(comparison_image, cmap="gray", vmin=0, vmax=1)
ax.set_title("Originale oben und Autoencoder-Rekonstruktionen unten")
ax.set_xlabel("Sechs Testbeispiele nebeneinander")
ax.set_ylabel("Bildzeile")
ax.set_xticks([])
ax.set_yticks([4, 13])
ax.set_yticklabels(["Original", "Rekonstruktion"])
plt.show()



## **3. Denoising und Grenzen**

### **3.1. Rauschen entfernen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_01.jpg?v=1784819126" width="250">



>* Verrauschte Eingaben sauber rekonstruieren
>* Wichtige Muster von Störungen trennen

>* Zufälliges Rauschen lässt sich oft entfernen
>* Mehrdeutige Störungen bleiben schwierig

>* Rekonstruktionen sind plausibel, nicht garantiert wahr
>* Denoising kann wichtige Details verändern



In [ ]:
#@title Python-Code - Rauschen entfernen

# Dieses Beispiel entfernt Rauschen aus Ziffernbildern.
# Ein kleiner Autoencoder lernt stabile Muster.
# Rekonstruktionsfehler zeigen Nutzen und Grenzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir nutzen kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(np.float32) / 16.0

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Ziffernbilder.")

# Für Denoising ist das Ziel das saubere Bild.
clean_data = images.reshape(len(images), -1)
rng = np.random.default_rng(42)

# Zufälliges Rauschen stört die Eingabe, nicht das Ziel.
noise = rng.normal(loc=0.0, scale=0.35, size=clean_data.shape)
noisy_data = np.clip(clean_data + noise, 0.0, 1.0)

# Der Split trennt Training und faire Auswertung.
noisy_train, noisy_test, clean_train, clean_test = train_test_split(
    noisy_data,
    clean_data,
    test_size=0.25,
    random_state=42,
)

# Der Autoencoder komprimiert über eine kleine versteckte Schicht.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16,),
    activation="relu",
    solver="adam",
    max_iter=120,
    random_state=42,
)

# Das Modell lernt verrauschte Eingaben sauber zu rekonstruieren.
autoencoder.fit(noisy_train, clean_train)
reconstructed_test = np.clip(autoencoder.predict(noisy_test), 0.0, 1.0)

# Kleinere Fehler bedeuten bessere Annäherung an saubere Bilder.
noisy_error = mean_squared_error(clean_test, noisy_test)
reconstructed_error = mean_squared_error(clean_test, reconstructed_test)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"MSE verrauschte Eingabe: {noisy_error:.4f}")
print(f"MSE nach Autoencoder: {reconstructed_error:.4f}")

# Ein einzelnes Bild zeigt Eingabe, Ziel und Rekonstruktion nebeneinander.
example_index = 0
comparison = np.hstack(
    [
        noisy_test[example_index].reshape(8, 8),
        clean_test[example_index].reshape(8, 8),
        reconstructed_test[example_index].reshape(8, 8),
    ]
)

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.imshow(comparison, cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Verrauscht | sauber | rekonstruiert")
ax.set_xlabel("Pixelpositionen in drei 8x8-Bildern")
ax.set_ylabel("Pixelzeile")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **3.2. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_02.jpg?v=1784819124" width="250">



>* Autoencoder lernen normale Muster zu rekonstruieren
>* Hoher Rekonstruktionsfehler weist auf Abweichungen hin

>* Rekonstruktionsfehler sind nur Warnsignale
>* Schwellenwerte und Fachprüfung bleiben entscheidend

>* Anomalien hängen stark vom Trainingskontext ab
>* Autoencoder liefern Hinweise, kein echtes Verständnis



In [ ]:
#@title Python-Code - Anomalien erkennen

# Dieses Beispiel erkennt auffällige Ziffern mit Rekonstruktionsfehlern.
# Ein kleiner Autoencoder entsteht hier durch PCA.
# Hohe Fehler markieren mögliche, aber unsichere Anomalien.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import sklearn

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Die Ziffernbilder haben nicht die erwartete Form.")

# Pixelwerte werden für das Modell auf null bis eins skaliert.
flat_images = images.reshape(len(images), -1) / 16.0
normal_mask = labels == 0
normal_images = flat_images[normal_mask]

# Der Autoencoder lernt nur normale Beispiele der Ziffer Null.
train_images, test_normal = train_test_split(
    normal_images, test_size=0.35, random_state=42
)

# PCA komprimiert und rekonstruiert wie ein sehr einfacher Autoencoder.
autoencoder = PCA(n_components=8, random_state=42)
autoencoder.fit(train_images)

# Wir vergleichen normale Nullen mit anderen Ziffern als Auffälligkeiten.
anomaly_images = flat_images[labels == 8][: len(test_normal)]
normal_recon = autoencoder.inverse_transform(autoencoder.transform(test_normal))
anomaly_recon = autoencoder.inverse_transform(autoencoder.transform(anomaly_images))

# Der mittlere quadratische Fehler misst die Rekonstruktionsqualität.
normal_errors = np.mean((test_normal - normal_recon) ** 2, axis=1)
anomaly_errors = np.mean((anomaly_images - anomaly_recon) ** 2, axis=1)
threshold = np.percentile(normal_errors, 95)

# Kurze Kennzahlen zeigen die Idee und ihre Grenze.
normal_flagged = np.mean(normal_errors > threshold)
anomaly_flagged = np.mean(anomaly_errors > threshold)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Schwellenwert aus normalen Nullen: {threshold:.4f}")
print(f"Normale Nullen über Schwelle: {normal_flagged:.1%}")
print(f"Andere Ziffern über Schwelle: {anomaly_flagged:.1%}")

# Ein Histogramm zeigt überlappende Fehler statt perfekte Trennung.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(normal_errors, bins=12, alpha=0.7, label="normale Nullen")
ax.hist(anomaly_errors, bins=12, alpha=0.7, label="andere Ziffern")
ax.axvline(threshold, color="black", linestyle="--", label="Schwelle")

# Beschriftungen helfen beim Interpretieren der Fehlerverteilung.
ax.set_title("Rekonstruktionsfehler als Anomalie-Hinweis")
ax.set_xlabel("Mittlerer quadratischer Rekonstruktionsfehler")
ax.set_ylabel("Anzahl Bilder")
ax.legend()
plt.show()



### **3.3. Eigenes Autoencoder Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_B/image_03_03.jpg?v=1784819128" width="250">



>* Klare Fragestellung und passende Daten festlegen
>* Rekonstruktionsfehler kritisch statt isoliert bewerten

>* Rekonstruktionsfehler visuell und fachlich prüfen
>* Anomaliehinweise vorsichtig und kontextbezogen deuten

>* Latenten Raum mit Interpolationen untersuchen
>* Generative Ergebnisse vorsichtig und kritisch bewerten



In [ ]:
#@title Python-Code - Eigenes Autoencoder Projekt

# Wir untersuchen Denoising mit einem kleinen Autoencoder.
# Der Engpass zeigt Grenzen der Rekonstruktion.
# Die Grafik vergleicht Rauschen und Fehler.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(float) / 16.0
labels = digits.target

# Diese Prüfung macht die Bildannahme sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine Bilder mit 8 mal 8 Pixeln.")

# Wir formen jedes Bild zu einer Pixelzeile um.
clean_data = images.reshape(len(images), 64)
train_clean, test_clean, train_labels, test_labels = train_test_split(
    clean_data, labels, test_size=0.25, random_state=42, stratify=labels
)

# Deterministisches Rauschen simuliert gestörte Eingaben.
rng = np.random.default_rng(42)
noise_level = 0.35
train_noisy = np.clip(train_clean + rng.normal(0, noise_level, train_clean.shape), 0, 1)
test_noisy = np.clip(test_clean + rng.normal(0, noise_level, test_clean.shape), 0, 1)

# Der Autoencoder lernt verrauschte Bilder sauber zu rekonstruieren.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16,), activation="relu", max_iter=120,
    random_state=42, solver="adam", learning_rate_init=0.01, verbose=False
)
autoencoder.fit(train_noisy, train_clean)

# Wir messen Fehler vor und nach der Rekonstruktion.
reconstructed = np.clip(autoencoder.predict(test_noisy), 0, 1)
noisy_error = mean_squared_error(test_clean, test_noisy)
reconstruction_error = mean_squared_error(test_clean, reconstructed)

# Ein untypisches Bild prüft eine Grenze des Modells.
odd_image = 1.0 - test_clean[0]
odd_reconstruction = np.clip(autoencoder.predict(odd_image.reshape(1, -1))[0], 0, 1)
odd_error = mean_squared_error(odd_image, odd_reconstruction)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"MSE verrauscht gegen sauber: {noisy_error:.4f}")
print(f"MSE rekonstruiert gegen sauber: {reconstruction_error:.4f}")
print(f"MSE untypische Eingabe gegen Rekonstruktion: {odd_error:.4f}")

# Eine Zeile zeigt Original, Rauschen, Rekonstruktion und Grenze.
example_index = 0
canvas = np.hstack([
    test_clean[example_index].reshape(8, 8),
    test_noisy[example_index].reshape(8, 8),
    reconstructed[example_index].reshape(8, 8),
    odd_image.reshape(8, 8),
    odd_reconstruction.reshape(8, 8),
])

fig, ax = plt.subplots(figsize=(9, 2.2))
ax.imshow(canvas, cmap="gray", vmin=0, vmax=1)
ax.set_title("Sauber | verrauscht | rekonstruiert | untypisch | Rekonstruktion")
ax.set_xlabel("Nebeneinander angeordnete 8x8-Bilder")
ax.set_ylabel("Pixelzeilen")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Autoencoder und Generierung**</font>


In this lecture, you learned to:
- Unterscheiden diskriminative und generative Aufgaben anhand kleiner Datenbeispiele. 
- Trainieren kleine Autoencoder zur Rekonstruktion von Digits- oder MNIST-Teilmengen. 
- Analysieren Rekonstruktionsfehler, Denoising, latente Interpolation und Grenzen generativer Ansätze. 

In the next Module (Module 20), we will go over 'Abschlussprojekt'